<a href="https://colab.research.google.com/github/JUSTBOOM03/Study-Log/blob/main/26_2_Wearable_Devices_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd
import numpy as np
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import StandardScaler
import os
import warnings
warnings.filterwarnings('ignore')

# 1. 파일 세팅 (5회차 완전 배제, Train: 1, 2, 3회차 / Test: 4회차)
train_scenarios = [
    {"base": "260603_공연1-1.txt", "task": "260603_백색소음1.txt", "label": 0, "session": "S1_1"},
    {"base": "260604_공연1-2.txt", "task": "260604_빠른음악1.txt", "label": 1, "session": "S1_2"},
    {"base": "260607_공연2-1.txt", "task": "260607_백색소음2.txt", "label": 0, "session": "S2_1"},
    {"base": "260608_공연2-2.txt", "task": "260608_빠른음악2.txt", "label": 1, "session": "S2_2"},
    {"base": "260609_공연3-1.txt", "task": "260609_백색소음3.txt", "label": 0, "session": "S3_1"},
    {"base": "260610_공연3-2.txt", "task": "260610_빠른음악3.txt", "label": 1, "session": "S3_2"},
    {"base": "260611_공연4-1.txt", "task": "260611_백색소음4.txt", "label": 0, "session": "S4_1"},
    {"base": "260613_공연4-2.txt", "task": "260613_빠른음악4.txt", "label": 1, "session": "S4_2"},


]

test_scenarios = [

    #{"base": "260614_공연5-1.txt", "task": "260614_백색소음5.txt", "label": 0, "session": "S5_1"},
    #{"base": "260615_공연5-2.txt", "task": "260615_빠른음악5.txt", "label": 1, "session": "S5_2"},
    {"base": "260616_공연6-1.txt", "task": "260616_백색소음6.txt", "label": 0, "session": "S6_1"},
    {"base": "260617_공연6-2.txt", "task": "260617_빠른음악6.txt", "label": 1, "session": "S6_2"},
]

# 2. 순수 6개 코어 피처 추출 함수
def extract_6features(filepath):
    try: df = pd.read_csv(filepath, sep='\t', header=None)
    except: return None

    # 기초 노이즈 제거 (1초 단위 중앙값 필터)
    df[1] = df[1].rolling(window=5, center=True, min_periods=1).median()
    df[2] = df[2].rolling(window=5, center=True, min_periods=1).median()

    window_rows, step_rows = 150, 50
    n_windows = (len(df) - window_rows) // step_rows + 1
    if n_windows <= 0: return None

    windows_data = []
    for i in range(n_windows):
        start = i * step_rows
        w = df.iloc[start:start + window_rows]
        hr, rr = w[1].values, w[2].values

        true_beats = [rr[0]]
        for j in range(1, len(rr)):
            if rr[j] != rr[j-1]: true_beats.append(rr[j])
        true_beats = np.array(true_beats)

        if len(true_beats) < 5: continue
        diff_rr = np.diff(true_beats)
        raw = w.iloc[:, 11:75].values.flatten()
        raw = raw[~np.isnan(raw)]

        windows_data.append({
            'Mean_HR': np.mean(hr),
            'Mean_RR': np.mean(true_beats),
            'SDNN': np.std(true_beats, ddof=1) if len(true_beats)>1 else 0,
            'RMSSD': np.sqrt(np.mean(diff_rr**2)) if len(diff_rr)>0 else 0,
            'Raw_Std': np.std(raw) if len(raw)>0 else 0,
            'Raw_Energy': np.sum(raw**2)/len(raw) if len(raw)>0 else 0
        })
    return pd.DataFrame(windows_data)

# 3. 1:1 베이스라인 차감(Delta) 정규화
def build_delta_matrix(scenarios):
    feature_cols = ['Mean_HR', 'Mean_RR', 'SDNN', 'RMSSD', 'Raw_Std', 'Raw_Energy']
    matrix_list = []

    for sc in scenarios:
        if not os.path.exists(sc["base"]) or not os.path.exists(sc["task"]):
            print(f"[경고] 파일 누락: {sc['base']} 또는 {sc['task']}")
            continue

        df_base = extract_6features(sc["base"])
        df_task = extract_6features(sc["task"])

        if df_base is not None and df_task is not None and not df_base.empty and not df_task.empty:
            base_means = df_base[feature_cols].mean()
            df_delta = df_task[feature_cols].copy()

            for col in feature_cols:
                df_delta[col] = df_delta[col] - base_means[col]

            df_delta['Label'] = sc["label"]
            matrix_list.append(df_delta)

    return pd.concat(matrix_list, ignore_index=True) if matrix_list else pd.DataFrame()

# 4. 파이프라인 가동 및 모델링
print("--- 데이터 로드 및 전처리 가동 ---")
train_df = build_delta_matrix(train_scenarios)
test_df  = build_delta_matrix(test_scenarios)

if not train_df.empty and not test_df.empty:
    X_train = train_df.drop(['Label'], axis=1)
    y_train = train_df['Label']
    X_test  = test_df.drop(['Label'], axis=1)
    y_test  = test_df['Label']

    print(f"\n[데이터 확인]")
    print(f"Train 샘플 수: {len(X_train)} (정상치: 949 내외)")
    print(f"Test 샘플 수: {len(X_test)} (정상치: 391 내외)")

    # 데이터 스케일링
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # 모델 1: SVM
    svm = SVC(kernel='rbf', random_state=42)
    svm.fit(X_train_scaled, y_train)
    svm_preds = svm.predict(X_test_scaled)

    # 모델 2: Random Forest
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    rf_preds = rf.predict(X_test)

    print(f"\n======================================")
    print(f"[최종 성적표] 오리지널 6 Features 기반")
    print(f"======================================")
    print(f"★ SVM 정확도: {accuracy_score(y_test, svm_preds):.4f}")
    print(classification_report(y_test, svm_preds, target_names=['백색소음(0)', '빠른음악(1)']))

    print(f"--------------------------------------")
    print(f"☆ Random Forest 정확도: {accuracy_score(y_test, rf_preds):.4f}")
    print(classification_report(y_test, rf_preds, target_names=['백색소음(0)', '빠른음악(1)']))
else:
    print("[오류] 파일 로드에 실패했습니다. 폴더 내 파일명과 코드를 다시 한번 확인해주세요.")

--- 데이터 로드 및 전처리 가동 ---

[데이터 확인]
Train 샘플 수: 1340 (정상치: 949 내외)
Test 샘플 수: 391 (정상치: 391 내외)

[최종 성적표] 오리지널 6 Features 기반
★ SVM 정확도: 0.7136
              precision    recall  f1-score   support

     백색소음(0)       0.68      0.79      0.73       190
     빠른음악(1)       0.76      0.64      0.70       201

    accuracy                           0.71       391
   macro avg       0.72      0.72      0.71       391
weighted avg       0.72      0.71      0.71       391

--------------------------------------
☆ Random Forest 정확도: 0.7161
              precision    recall  f1-score   support

     백색소음(0)       0.68      0.78      0.73       190
     빠른음악(1)       0.76      0.66      0.70       201

    accuracy                           0.72       391
   macro avg       0.72      0.72      0.72       391
weighted avg       0.72      0.72      0.72       391

